# One complete study: from LoRaWAN receptions to an SNR forecast
**Outcome:** reproduce a six-hour sensor01 SNR study, explain each decision, and adapt the contract.
**Prerequisites:** notebook 00, regression and chronological validation; notebook 04 explains the neural training loop.
**Published replay contract:** sensor01–gatewayA, 24 closed hourly RSSI/SNR values, the mean SNR of the single hour closing six hours later, future eligible hours on the same link.
**Evidence status:** a teaching replay of the existing controlled anchor, not a new benchmark or independent deployment. Work through the numbered steps before the adaptation exercise.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd() if (Path.cwd()/'data/manifest.json').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/'scripts'))
from revision_models import fit, configurations, Network
from running_snr_example import verify, verify_protocol
from learning_checks import check_first_forecast, check_label_boundary

MODE = "reproduce"  # Change to "adapt" for a different history or horizon.
WINDOW, HORIZON = 24, 6
assert MODE in ["reproduce", "adapt"]
if MODE == "reproduce" and (WINDOW, HORIZON) != (24, 6):
    raise ValueError("Use adaptation mode when changing the prediction contract.")


## Before the dataset: one complete prediction
Suppose three consecutive closed hourly SNR means are **2, 4, 9 dB**, and the next hour later has SNR **5 dB**. Persistence predicts 9; the historical mean predicts 5. Their absolute errors are 4 and 0 dB. The future outcome is used only to score the forecasts. These numbers are illustrative.

Now hide this explanation and reconstruct both predictions. This small calculation is the same reference calculation used on the public data.

In [ ]:
# Replace None with your answers; an unanswered exercise is not a pass.
my_persistence, my_mean = None, None
print(check_first_forecast(my_persistence, my_mean))


## Complete one missing step
An origin is **2025-03-06 20:00 UTC**, the horizon is **3 h**, and validation begins at **23:00 UTC**. Complete the label time and decide whether the origin belongs in training. Explain the boundary rule before asking for feedback.

In [ ]:
my_label_time, my_training_choice = None, None
print(check_label_boundary('2025-03-06 20:00Z', 3, '2025-03-06 23:00Z',
    my_label_time, my_training_choice))


<details><summary>Worked reasoning after your attempt</summary>

The label arrives at 23:00 UTC, exactly when validation begins. The stated protocol requires it strictly earlier, so this origin is purged from training. A rule that checks only `origin < validation_start` would admit it incorrectly.

</details>

## Diagnose a plausible error
A colleague uses `origin < validation_start` as the entire training rule. Identify the missing condition, then give an origin for which it changes membership. The real-data code below will let you check your answer.

The setup has two modes. **reproduce** checks the published 24 h / 6 h study. **adapt** checks the new contract and writes a separate reader result; it never requires the original scores. Automated execution of this notebook leaves unanswered reader prompts marked `not_attempted`.

## 1. Inspect reception records and their units
The following function makes every aggregation step explicit. RSSI and SNR get separate nonmissing counts. An hourly timestamp denotes when that hour has closed; an empty hour remains on the calendar.

In [ ]:
def load_records(root):
    raw=pd.read_parquet(root/'data/raw/uva/sensor01_gatewayA.parquet')
    receptions=raw.set_index('Timestamp').sort_index()
    hourly=receptions[['RSSI (dBm)','SNR (dB)']].rename(
        columns={'RSSI (dBm)':'rssi','SNR (dB)':'snr'}).resample('1h').mean()
    hourly['rssi_count']=receptions['RSSI (dBm)'].resample('1h').count()
    hourly['snr_count']=receptions['SNR (dB)'].resample('1h').count()
    # A value stamped 20:00 summarizes [19:00, 20:00), in UTC.
    hourly.index=hourly.index+pd.Timedelta(hours=1)
    return raw,hourly

raw,hourly=load_records(ROOT)
display(raw[['Timestamp','RSSI (dBm)','SNR (dB)','Counter (-)']].head())
display(hourly.head())
assert hourly.index.to_series().diff().dropna().eq(pd.Timedelta(hours=1)).all()
print('Source receptions:',len(raw),'Calendar hours:',len(hourly))


## 2. Build inputs, the future target and chronological partitions
Keep the 60/20/20 boundaries on the original calendar. Shift by elapsed hours, then exclude incomplete histories/outcomes. A training label must arrive strictly before validation starts. A future missing label prevents scoring an outcome; it does not prevent issuing a forecast from an already complete history.

**Before running:** explain why shifting after dropping empty hours changes the target.

In [ ]:
def make_examples(hourly,window=24,horizon=6):
    if not isinstance(window,int) or not isinstance(horizon,int) or min(window,horizon)<1:
        raise ValueError('History and horizon must be positive integer hours.')
    frame=pd.DataFrame(index=hourly.index)
    for variable in ['rssi','snr']:
        for lag in range(window):
            frame[f'{variable}_lag{lag}']=hourly[variable].shift(lag)
    features=list(frame.columns)
    frame['truth']=hourly.snr.shift(-horizon)
    frame['label_available_at']=frame.index+pd.Timedelta(hours=horizon)
    c1,c2=[hourly.index[int(f*len(hourly))] for f in [.6,.8]]
    end=hourly.index[-1]+pd.Timedelta(hours=1)
    history_ready=frame[features].notna().all(axis=1)
    eligible=history_ready & frame.truth.notna()
    frame['split']='excluded'
    for name,mask in [('train',(frame.index<c1)&(frame.label_available_at<c1)),
                      ('validation',(frame.index>=c1)&(frame.index<c2)&(frame.label_available_at<c2)),
                      ('test',(frame.index>=c2)&(frame.label_available_at<end))]:
        frame.loc[mask & eligible,'split']=name
    return frame,features,dict(train_end=str(c1),validation_end=str(c2),test_end=str(end))

frame,features,boundaries=make_examples(hourly,window=WINDOW,horizon=HORIZON)
display(frame.groupby('split').size().rename('origins'))
print(boundaries)
protocol=verify_protocol(hourly,frame,features,boundaries,WINDOW,HORIZON)
print(protocol)
assert frame.loc[frame.split.eq('train'),'label_available_at'].max()<pd.Timestamp(boundaries['train_end'])
assert frame.loc[frame.split.eq('validation'),'label_available_at'].max()<pd.Timestamp(boundaries['validation_end'])


## 3. Hand-check the first eligible test window
This origin is selected by time, before inspecting errors. Read all rows in the declared history. The latest hour contains several receptions: the first three raw rows are not the full mean. The target row is visible only for retrospective checking.

In [ ]:
origin=frame.index[frame.split.eq('test')][0]
history=hourly.loc[origin-pd.Timedelta(hours=WINDOW-1):origin]
latest=raw[(raw.Timestamp>=origin-pd.Timedelta(hours=1))&(raw.Timestamp<origin)]
display(latest[['Timestamp','RSSI (dBm)','SNR (dB)']].head(3))
display(history)
assert len(history)==WINDOW
np.testing.assert_allclose(history.iloc[-1].snr,latest['SNR (dB)'].mean())
np.testing.assert_allclose(frame.loc[origin,features[:WINDOW]],history.rssi.iloc[::-1])
print('Target availability:',frame.loc[origin,'label_available_at'])
print('Later observed target:',frame.loc[origin,'truth'])


## 4. Compute references before learning
Persistence repeats the latest SNR. The historical mean averages the closed SNR values in the declared history. Calculate both and their single-example errors; this example does not determine a population ranking.

In [ ]:
truth=frame.loc[origin,'truth']
references=pd.Series({'persistence':history.snr.iloc[-1],'window_mean':history.snr.mean()})
display(pd.DataFrame({'prediction_dB':references,'absolute_error_dB':(references-truth).abs()}))
print('Ready-to-issue test histories:',int(frame.loc[frame.index>=pd.Timestamp(boundaries['validation_end']),features].notna().all(axis=1).sum()))
print('Evaluable test origins:',int(frame.split.eq('test').sum()))


## 5. Select Ridge and MLP on validation
The visible search below passes only training and validation arrays to `fit`. It tests the original six Ridge penalties and four MLP settings, with seed 42. `revision_models.py` supplies the same training implementation explained in notebook 04: training-only scaling, Adam, scaled squared loss, validation MAE in dB, clipping and early stopping. No test outcome participates in this search.

The selected MLP has two hidden layers with ReLU. Print the architecture and inspect how the declared inputs become one output. A candidate grid is a declared comparison budget, not a guarantee of optimal tuning.

In [ ]:
def mae(truth,prediction):
    return float(np.abs(np.asarray(truth)-np.asarray(prediction)).mean())

def select_on_validation(frame,features,window=24):
    # Test rows are not passed to fitting, stopping or candidate selection.
    tr=frame[frame.split.eq('train')];va=frame[frame.split.eq('validation')]
    models={};selected={};candidates=[]
    for family in ['ridge','mlp']:
        best=None
        for candidate,config in enumerate(configurations(family)):
            fitted,history,info=fit(tr[features].to_numpy(),tr[['truth']].to_numpy(),
                va[features].to_numpy(),va[['truth']].to_numpy(),family=family,
                config=config,seed=42,score=mae,window=window)
            value=mae(va[['truth']].to_numpy(),fitted.predict(va[features].to_numpy()))
            candidates.append(dict(model=family,candidate=candidate,config=json.dumps(config,sort_keys=True),validation_mae=value))
            if best is None or value<best[0]:best=(value,fitted,config,history)
        models[family]=best[1];selected[family]=best[2]
    return models,selected,pd.DataFrame(candidates)

models,selected,candidates=select_on_validation(frame,features,window=WINDOW)
display(candidates)
print('Selected settings:',selected)
print(models['mlp'].model)
if MODE == 'reproduce':
    assert selected['ridge']=={'alpha':1000.0}
    assert selected['mlp']=={'width':32,'lr':0.001,'decay':0.01}


## 6. Freeze the choice and evaluate on identical test origins
Every procedure uses the same test examples. Positive ΔMAE against the historical mean favors that simple reference. RMSE answers a related but different loss question; choose the primary metric before looking at the test.

In [ ]:
def evaluate(frame,features,models,window=24):
    test=frame[frame.split.eq('test')]
    predictions=pd.DataFrame(index=test.index)
    predictions['truth']=test.truth
    predictions['persistence']=test.snr_lag0
    predictions['window_mean']=test[[f'snr_lag{k}' for k in range(window)]].mean(axis=1)
    for family,fitted in models.items():predictions[family]=fitted.predict(test[features].to_numpy()).ravel()
    rows=[]
    for model in ['persistence','window_mean','ridge','mlp']:
        error=predictions[model]-predictions.truth
        rows.append(dict(model=model,mae=float(error.abs().mean()),rmse=float(np.sqrt((error**2).mean()))))
    metrics=pd.DataFrame(rows)
    metrics['mae_minus_mean']=metrics.mae-float(metrics.loc[metrics.model.eq('window_mean'),'mae'].iloc[0])
    return predictions,metrics

predictions,metrics=evaluate(frame,features,models,window=WINDOW)
display(metrics)
if MODE == 'reproduce':
    audit=verify(ROOT,frame,features,selected,predictions)
else:
    audit=verify_protocol(hourly,frame,features,boundaries,WINDOW,HORIZON)
    destination=ROOT/'results/reader_runs'/f'sensor01_w{WINDOW}_h{HORIZON}'
    destination.mkdir(parents=True,exist_ok=True)
    predictions.to_csv(destination/'predictions.csv.gz',index_label='origin')
    metrics.to_csv(destination/'metrics.csv',index=False)
    candidates.to_csv(destination/'candidates.csv',index=False)
    (destination/'contract.json').write_text(json.dumps(dict(window=WINDOW,horizon=HORIZON,
        boundaries=boundaries,selected=selected,protocol=audit,
        evidence='Reader adaptation; not a published comparison'),indent=2))
    print('Adaptation saved to',destination)
print(audit)
assert audit['status']=='passed' 


## 7. Diagnose failures without selecting on test errors
The five days with the largest MLP absolute error are post-hoc diagnostic slices. Their counts and paired reference errors matter. They are not additional independent trials and do not by themselves identify a cause.

In [ ]:
errors=predictions.drop(columns='truth').sub(predictions.truth,axis=0).abs()
daily=errors.groupby(errors.index.floor('D')).mean()
daily['origins']=errors.groupby(errors.index.floor('D')).size()
display(daily.sort_values('mlp',ascending=False).head(5))
cutoff=pd.Timestamp(boundaries['train_end'])
changed=hourly.copy();changed.loc[changed.index>=cutoff,['rssi','snr']]+=100
perturbed,_,_=make_examples(changed,window=WINDOW,horizon=HORIZON)
past=frame.label_available_at<cutoff
np.testing.assert_allclose(frame.loc[past,features+['truth']],perturbed.loc[past,features+['truth']],equal_nan=True)
print('Future perturbation leaves all earlier labeled examples unchanged.')


## 8. State the conclusion in three parts
**For the published six-hour replay:** the 24-hour mean has lower test MAE than the selected Ridge and teaching MLP fit; the learned procedures have slightly lower RMSE. In adaptation mode, write this sentence again from your own output.

**Transferable lesson:** evaluate added model structure against more than one credible reference, on the same population and under a declared primary loss. This outcome does not justify choosing a favorable initialization after testing.

**Not established:** the observed errors do not certify transfer to another link, performance with missing histories, packet delivery, or an operational intervention. The full manuscript reports the other declared initializations and sensitivity studies.

**Independent adaptation:** set `MODE = "adapt"` and `HORIZON = 12` in the setup cell, then restart and run the notebook. Keep `WINDOW = 24` for this exercise. The protocol checks still apply; equality with the published six-hour scores and selected settings does not. Before fitting, predict which columns, availability timestamps, phase purges and evaluation counts must change. Implement that new contract in a fresh result record. Do not claim that a score difference isolates the horizon if the eligible population changes. Then use notebook 09 to adapt an unfamiliar CSV schema.

**Checkpoint:** if RMSE were the actual deployment objective, at which earlier step would the selection criterion change?

<details><summary>Compare your reasoning</summary>

Specify the loss during protocol definition and use it for validation selection before inspecting held-out RMSE. Choosing the most favorable test metric after seeing results changes the question retrospectively.

</details>